[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/optimization/03_gradient_descent_and_convergence/first_principles.ipynb)

# Topic 03: Gradient Descent & Convergence Theory

## 1. First-Principles Intuition & Motivation

### 1.1 A Landscape Probed Only Locally

Imagine standing on a foggy mountainside, able to sense only the slope of the ground beneath your feet.
The natural strategy is to step downhill, repeatedly. Optimization formalizes this: a differentiable
objective $f: \mathbb{R}^n \to \mathbb{R}$ reveals, at any query point $\mathbf{x}$, exactly two pieces of
local data — the height $f(\mathbf{x})$ and the slope $\nabla f(\mathbf{x})$.

Which direction is *steepest*? The first-order Taylor expansion answers this. For a unit direction
$\mathbf{d}$ and small step $\alpha \gt 0$,

$$
f(\mathbf{x} + \alpha \mathbf{d}) \approx f(\mathbf{x}) + \alpha \nabla f(\mathbf{x})^T \mathbf{d}
$$

so the instantaneous rate of change is the inner product $\nabla f(\mathbf{x})^T \mathbf{d}$. By the
Cauchy-Schwarz inequality, over all unit vectors this inner product is minimized (most negative) when
$\mathbf{d}$ points exactly opposite the gradient:

$$
\mathbf{d}^* = -\frac{\nabla f(\mathbf{x})}{\lVert \nabla f(\mathbf{x})\rVert}
$$

This single observation — *the negative gradient is the locally steepest descent direction* — generates
the entire family of first-order methods.

### 1.2 The Two Constants That Govern Everything

A slope tells you nothing about how far the slope remains valid. Convergence theory therefore rests on
two curvature bounds:

- **Smoothness constant $L$** (upper curvature): the gradient cannot change faster than $L$ per unit of
  distance. It tells us how far we may trust the linear model, and hence how large a step is safe.
- **Strong convexity constant $\mu$** (lower curvature): the function bends upward at least this fast in
  every direction, so the landscape is a genuine bowl and the gradient magnitude reflects distance to
  the bottom.

Their ratio $\kappa = L/\mu \ge 1$ is the **condition number**. When $\kappa = 1$ the level sets are
spheres and one well-chosen step reaches the minimizer. When $\kappa$ is large the level sets are long
thin ellipsoids: any single step size is simultaneously too large for the stiff direction (causing
oscillation across the valley) and too small for the flat direction (causing a slow crawl along it).
This is the famous **zig-zagging** of gradient descent on ill-conditioned problems, and every rate we
prove will be a function of $\kappa$.

### 1.3 The Continuous-Time View: Gradient Flow

Shrinking the step size to zero, the iterates of gradient descent trace the solution of the
**gradient flow** ordinary differential equation

$$
\dot{\mathbf{x}}(t) = -\nabla f(\mathbf{x}(t))
$$

Along this flow the objective is a Lyapunov function:

$$
\frac{d}{dt} f(\mathbf{x}(t)) = \nabla f(\mathbf{x}(t))^T \dot{\mathbf{x}}(t) = -\lVert \nabla f(\mathbf{x}(t))\rVert^2 \le 0
$$

so the continuous dynamics *always* descend. Gradient descent with step $\alpha$ is precisely the
forward Euler discretization $\mathbf{x}_{k+1} = \mathbf{x}_k + \alpha \dot{\mathbf{x}}_k$, and the
step-size restrictions we derive below ($\alpha \lt 2/L$) are exactly the numerical stability limits of
that discretization. Momentum methods, in turn, discretize a *second-order* ODE — a ball with inertia
rolling through the landscape under friction — which explains both their speed and their oscillations.
This ODE dictionary (algorithms as discretized dynamics) is one of the most productive bridges between
optimization and physics.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (Gradient descent).** Given $f \in \mathcal{C}^1(\mathbb{R}^n)$, an initial point
$\mathbf{x}_0$, and step sizes $\alpha_k \gt 0$, the gradient descent iteration is

$$
\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha_k \nabla f(\mathbf{x}_k), \qquad k = 0, 1, 2, \dots
$$

**Definition 2 (Steepest descent direction).** At a point $\mathbf{x}$ with $\nabla f(\mathbf{x}) \neq \mathbf{0}$,
the steepest descent direction is the solution of

$$
\mathbf{d}^* = \arg\min_{\lVert \mathbf{d}\rVert_2 = 1} \nabla f(\mathbf{x})^T \mathbf{d}
$$

*Derivation.* By Cauchy-Schwarz, $\nabla f(\mathbf{x})^T \mathbf{d} \ge -\lVert \nabla f(\mathbf{x})\rVert_2 \lVert \mathbf{d}\rVert_2 = -\lVert \nabla f(\mathbf{x})\rVert_2$,
with equality if and only if $\mathbf{d}$ is a negative multiple of $\nabla f(\mathbf{x})$. Hence
$\mathbf{d}^* = -\nabla f(\mathbf{x}) / \lVert \nabla f(\mathbf{x})\rVert_2$, justifying Definition 1
as the method of steepest descent (in the Euclidean norm).

**Definition 3 ($L$-smoothness).** A differentiable function $f: \mathbb{R}^n \to \mathbb{R}$ is
**$L$-smooth** if its gradient is $L$-Lipschitz:

$$
\lVert \nabla f(\mathbf{x}) - \nabla f(\mathbf{y})\rVert \le L \lVert \mathbf{x} - \mathbf{y}\rVert \qquad \text{for all } \mathbf{x}, \mathbf{y} \in \mathbb{R}^n
$$

Equivalent characterizations (for $f \in \mathcal{C}^2$):

- $\nabla^2 f(\mathbf{x}) \preceq L I$ for all $\mathbf{x}$, i.e. every Hessian eigenvalue is at most $L$;
- the quadratic upper bound (descent lemma, Theorem 1 below) holds globally.

For a quadratic $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^T A \mathbf{x} - \mathbf{b}^T \mathbf{x}$ with
$A$ symmetric positive semidefinite, $L = \lambda_{\max}(A)$. For least squares
$f(\mathbf{w}) = \frac{1}{2}\lVert X\mathbf{w} - \mathbf{y}\rVert^2$, $L = \lambda_{\max}(X^T X)$.

**Definition 4 ($\mu$-strong convexity).** A differentiable $f$ is **$\mu$-strongly convex**
($\mu \gt 0$) if

$$
f(\mathbf{y}) \ge f(\mathbf{x}) + \nabla f(\mathbf{x})^T(\mathbf{y} - \mathbf{x}) + \frac{\mu}{2}\lVert \mathbf{y} - \mathbf{x}\rVert^2 \qquad \text{for all } \mathbf{x}, \mathbf{y}
$$

For $f \in \mathcal{C}^2$ this is equivalent to $\nabla^2 f(\mathbf{x}) \succeq \mu I$. Strong convexity
guarantees a unique minimizer $\mathbf{x}^*$ and links gradient size to suboptimality and distance:

$$
f(\mathbf{x}) - f^* \le \frac{1}{2\mu}\lVert \nabla f(\mathbf{x})\rVert^2, \qquad \lVert \mathbf{x} - \mathbf{x}^*\rVert \le \frac{1}{\mu}\lVert \nabla f(\mathbf{x})\rVert
$$

**Definition 5 (Condition number).** For an $L$-smooth, $\mu$-strongly convex function, the condition
number is

$$
\kappa = \frac{L}{\mu} \ge 1
$$

For a quadratic with Hessian $A \succ 0$, $\kappa = \lambda_{\max}(A)/\lambda_{\min}(A)$, the ratio of
extreme curvatures — geometrically, the squared aspect ratio of the elliptical level sets.

**Definition 6 (Polyak-Lojasiewicz inequality).** A differentiable function $f$ with minimum value
$f^*$ satisfies the **PL inequality** with constant $\mu \gt 0$ if

$$
\frac{1}{2}\lVert \nabla f(\mathbf{x})\rVert^2 \ge \mu \left( f(\mathbf{x}) - f^* \right) \qquad \text{for all } \mathbf{x}
$$

Three facts make PL remarkable:

1. **Strong convexity implies PL** with the same $\mu$ (minimize both sides of Definition 4 over $\mathbf{y}$).
2. **PL does not imply convexity.** The function $f(x) = x^2 + 3\sin^2 x$ is non-convex yet PL, and
   over-parameterized least squares (singular $X^T X$ with consistent data) is PL but not strongly convex.
3. PL is exactly what the linear-rate proof consumes (Theorem 5): every stationary point of a PL
   function is a global minimizer.

**Theorem 1 (Descent lemma / quadratic upper bound).** If $f$ is $L$-smooth, then for all
$\mathbf{x}, \mathbf{y} \in \mathbb{R}^n$:

$$
f(\mathbf{y}) \le f(\mathbf{x}) + \nabla f(\mathbf{x})^T (\mathbf{y} - \mathbf{x}) + \frac{L}{2}\lVert \mathbf{y} - \mathbf{x}\rVert^2
$$

**Theorem 2 (Sufficient decrease and non-convex rate).** Let $f$ be $L$-smooth and bounded below by
$f^*$. Gradient descent with fixed step $\alpha = 1/L$ satisfies, for every $k$,

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \frac{1}{2L}\lVert \nabla f(\mathbf{x}_k)\rVert^2
$$

and consequently after $k$ iterations

$$
\min_{0 \le i \lt k} \lVert \nabla f(\mathbf{x}_i)\rVert \le \sqrt{\frac{2L\left(f(\mathbf{x}_0) - f^*\right)}{k}} = O\!\left(\frac{1}{\sqrt{k}}\right)
$$

No convexity is assumed: this is the guarantee that underlies gradient-based training of deep networks.

**Theorem 3 (Convex sublinear rate).** If $f$ is convex, $L$-smooth, with minimizer $\mathbf{x}^*$,
then gradient descent with $\alpha = 1/L$ satisfies

$$
f(\mathbf{x}_k) - f^* \le \frac{L \lVert \mathbf{x}_0 - \mathbf{x}^*\rVert^2}{2k} = O\!\left(\frac{1}{k}\right)
$$

**Theorem 4 (Strongly convex linear rate).** If $f$ is $L$-smooth and $\mu$-strongly convex, gradient
descent with $\alpha = 1/L$ converges linearly (geometrically):

$$
f(\mathbf{x}_k) - f^* \le \left(1 - \frac{\mu}{L}\right)^k \left(f(\mathbf{x}_0) - f^*\right)
$$

Reaching accuracy $\epsilon$ therefore costs $O(\kappa \log(1/\epsilon))$ iterations.

**Theorem 5 (PL linear rate).** The same conclusion as Theorem 4 holds if strong convexity is replaced
by the PL inequality with constant $\mu$ — convexity is not needed.

**Theorem 6 (Exact quadratic rate).** For $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^T A\mathbf{x} - \mathbf{b}^T\mathbf{x}$
with $\mu I \preceq A \preceq L I$, the optimal fixed step is $\alpha^* = 2/(L + \mu)$, achieving the
per-iterate contraction

$$
\lVert \mathbf{x}_{k+1} - \mathbf{x}^*\rVert \le \frac{\kappa - 1}{\kappa + 1} \lVert \mathbf{x}_k - \mathbf{x}^*\rVert
$$

**Definition 7 (Heavy-ball momentum, Polyak).** With momentum coefficient $\beta \in [0, 1)$:

$$
\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha \nabla f(\mathbf{x}_k) + \beta\left(\mathbf{x}_k - \mathbf{x}_{k-1}\right)
$$

**Definition 8 (Nesterov accelerated gradient).** Evaluate the gradient at a *look-ahead* point:

$$
\mathbf{y}_k = \mathbf{x}_k + \beta_k\left(\mathbf{x}_k - \mathbf{x}_{k-1}\right), \qquad \mathbf{x}_{k+1} = \mathbf{y}_k - \alpha \nabla f(\mathbf{y}_k)
$$

**Theorem 7 (Acceleration).** For convex $L$-smooth $f$, Nesterov's method with $\alpha = 1/L$ and a
suitable $\beta_k$ sequence achieves

$$
f(\mathbf{x}_k) - f^* \le \frac{2L\lVert \mathbf{x}_0 - \mathbf{x}^*\rVert^2}{(k+1)^2} = O\!\left(\frac{1}{k^2}\right)
$$

which matches the $\Omega(1/k^2)$ lower bound for first-order methods (Nesterov). Under $\mu$-strong
convexity, heavy-ball parameters $\alpha = 4/(\sqrt{L} + \sqrt{\mu})^2$ and
$\beta = \left((\sqrt{\kappa}-1)/(\sqrt{\kappa}+1)\right)^2$ give, on quadratics, the accelerated
contraction factor

$$
\frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1} \quad \text{per iteration, i.e. } O(\sqrt{\kappa}\log(1/\epsilon)) \text{ iterations}
$$

versus $O(\kappa \log(1/\epsilon))$ for plain gradient descent — a quadratic improvement in $\kappa$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1: The Descent Lemma from the Fundamental Theorem of Calculus

**Claim.** $L$-smoothness implies $f(\mathbf{y}) \le f(\mathbf{x}) + \nabla f(\mathbf{x})^T(\mathbf{y}-\mathbf{x}) + \frac{L}{2}\lVert \mathbf{y}-\mathbf{x}\rVert^2$.

**Step 1.** Define the 1D restriction $g(t) = f(\mathbf{x} + t(\mathbf{y}-\mathbf{x}))$ for $t \in [0,1]$.
By the chain rule $g'(t) = \nabla f(\mathbf{x} + t(\mathbf{y}-\mathbf{x}))^T(\mathbf{y}-\mathbf{x})$.

**Step 2.** The fundamental theorem of calculus gives

$$
f(\mathbf{y}) - f(\mathbf{x}) = g(1) - g(0) = \int_0^1 \nabla f(\mathbf{x} + t(\mathbf{y}-\mathbf{x}))^T(\mathbf{y}-\mathbf{x})\, dt
$$

**Step 3.** Add and subtract the constant vector $\nabla f(\mathbf{x})$ inside the integral:

$$
f(\mathbf{y}) - f(\mathbf{x}) = \nabla f(\mathbf{x})^T(\mathbf{y}-\mathbf{x}) + \int_0^1 \left[\nabla f(\mathbf{x} + t(\mathbf{y}-\mathbf{x})) - \nabla f(\mathbf{x})\right]^T(\mathbf{y}-\mathbf{x})\, dt
$$

**Step 4.** Bound the integrand by Cauchy-Schwarz and then by $L$-Lipschitzness of the gradient:

$$
\left[\nabla f(\mathbf{x} + t(\mathbf{y}-\mathbf{x})) - \nabla f(\mathbf{x})\right]^T(\mathbf{y}-\mathbf{x}) \le L t \lVert \mathbf{y}-\mathbf{x}\rVert \cdot \lVert \mathbf{y}-\mathbf{x}\rVert = L t \lVert \mathbf{y}-\mathbf{x}\rVert^2
$$

**Step 5.** Integrate: $\int_0^1 L t \lVert \mathbf{y}-\mathbf{x}\rVert^2 dt = \frac{L}{2}\lVert \mathbf{y}-\mathbf{x}\rVert^2$. Therefore

$$
\boxed{f(\mathbf{y}) \le f(\mathbf{x}) + \nabla f(\mathbf{x})^T(\mathbf{y}-\mathbf{x}) + \frac{L}{2}\lVert \mathbf{y}-\mathbf{x}\rVert^2}
$$

### Proof 2: Sufficient Decrease and the $O(1/\sqrt{k})$ Non-Convex Rate

**Claim.** For $L$-smooth $f$ bounded below by $f^*$, gradient descent with $\alpha = 1/L$ satisfies
$\min_{0 \le i \lt k}\lVert \nabla f(\mathbf{x}_i)\rVert^2 \le 2L(f(\mathbf{x}_0)-f^*)/k$.

**Step 1 (sufficient decrease).** Apply the descent lemma with $\mathbf{y} = \mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\nabla f(\mathbf{x}_k)$,
so $\mathbf{y} - \mathbf{x} = -\alpha \nabla f(\mathbf{x}_k)$:

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \alpha \lVert \nabla f(\mathbf{x}_k)\rVert^2 + \frac{L\alpha^2}{2}\lVert \nabla f(\mathbf{x}_k)\rVert^2 = f(\mathbf{x}_k) - \alpha\left(1 - \frac{L\alpha}{2}\right)\lVert \nabla f(\mathbf{x}_k)\rVert^2
$$

**Step 2.** With $\alpha = 1/L$ the factor becomes $\frac{1}{L}\left(1 - \frac{1}{2}\right) = \frac{1}{2L}$:

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \frac{1}{2L}\lVert \nabla f(\mathbf{x}_k)\rVert^2
$$

Each step pays for itself in squared gradient norm.

**Step 3 (telescoping).** Sum the inequality over $i = 0, \dots, k-1$; the left side telescopes:

$$
\frac{1}{2L}\sum_{i=0}^{k-1} \lVert \nabla f(\mathbf{x}_i)\rVert^2 \le f(\mathbf{x}_0) - f(\mathbf{x}_k) \le f(\mathbf{x}_0) - f^*
$$

**Step 4.** The minimum of $k$ numbers is at most their average:

$$
\min_{0 \le i \lt k} \lVert \nabla f(\mathbf{x}_i)\rVert^2 \le \frac{1}{k}\sum_{i=0}^{k-1}\lVert \nabla f(\mathbf{x}_i)\rVert^2 \le \frac{2L\left(f(\mathbf{x}_0) - f^*\right)}{k}
$$

Taking square roots yields the rate. Note $\sum_i \lVert \nabla f(\mathbf{x}_i)\rVert^2 \lt \infty$, so
$\nabla f(\mathbf{x}_k) \to \mathbf{0}$: gradient descent finds approximate stationary points of *any*
smooth bounded function.

$$
\boxed{\min_{0 \le i \lt k} \lVert \nabla f(\mathbf{x}_i)\rVert \le \sqrt{\frac{2L\left(f(\mathbf{x}_0) - f^*\right)}{k}}}
$$

### Proof 3: The $O(1/k)$ Rate for Convex $L$-Smooth Functions

**Claim.** For convex $L$-smooth $f$ with minimizer $\mathbf{x}^*$, gradient descent with $\alpha = 1/L$
satisfies $f(\mathbf{x}_k) - f^* \le \frac{L\lVert \mathbf{x}_0 - \mathbf{x}^*\rVert^2}{2k}$.

**Step 1 (distance decrease identity).** Expand the squared distance after one step,
writing $\mathbf{g}_i = \nabla f(\mathbf{x}_i)$:

$$
\lVert \mathbf{x}_{i+1} - \mathbf{x}^*\rVert^2 = \lVert \mathbf{x}_i - \mathbf{x}^*\rVert^2 - 2\alpha\, \mathbf{g}_i^T(\mathbf{x}_i - \mathbf{x}^*) + \alpha^2 \lVert \mathbf{g}_i\rVert^2
$$

**Step 2 (convexity bounds the cross term).** Convexity gives
$f^* \ge f(\mathbf{x}_i) + \mathbf{g}_i^T(\mathbf{x}^* - \mathbf{x}_i)$, i.e.
$\mathbf{g}_i^T(\mathbf{x}_i - \mathbf{x}^*) \ge f(\mathbf{x}_i) - f^*$. Also, sufficient decrease
(Proof 2, Step 2) gives $\lVert \mathbf{g}_i\rVert^2 \le 2L\left(f(\mathbf{x}_i) - f(\mathbf{x}_{i+1})\right)$.

**Step 3.** Substitute both with $\alpha = 1/L$:

$$
\lVert \mathbf{x}_{i+1} - \mathbf{x}^*\rVert^2 \le \lVert \mathbf{x}_i - \mathbf{x}^*\rVert^2 - \frac{2}{L}\left(f(\mathbf{x}_i) - f^*\right) + \frac{2}{L}\left(f(\mathbf{x}_i) - f(\mathbf{x}_{i+1})\right)
$$

$$
\lVert \mathbf{x}_{i+1} - \mathbf{x}^*\rVert^2 \le \lVert \mathbf{x}_i - \mathbf{x}^*\rVert^2 - \frac{2}{L}\left(f(\mathbf{x}_{i+1}) - f^*\right)
$$

**Step 4 (telescoping).** Sum over $i = 0, \dots, k-1$ and drop the non-negative final distance:

$$
\frac{2}{L}\sum_{i=1}^{k}\left(f(\mathbf{x}_i) - f^*\right) \le \lVert \mathbf{x}_0 - \mathbf{x}^*\rVert^2
$$

**Step 5.** By sufficient decrease the sequence $f(\mathbf{x}_i)$ is non-increasing, so each of the $k$
summands is at least $f(\mathbf{x}_k) - f^*$:

$$
k\left(f(\mathbf{x}_k) - f^*\right) \le \sum_{i=1}^{k}\left(f(\mathbf{x}_i) - f^*\right) \le \frac{L}{2}\lVert \mathbf{x}_0 - \mathbf{x}^*\rVert^2
$$

$$
\boxed{f(\mathbf{x}_k) - f^* \le \frac{L\lVert \mathbf{x}_0 - \mathbf{x}^*\rVert^2}{2k}}
$$

### Proof 4: Linear Rate $(1 - \mu/L)^k$ Under Strong Convexity

**Claim.** For $L$-smooth, $\mu$-strongly convex $f$, gradient descent with $\alpha = 1/L$ satisfies
$f(\mathbf{x}_k) - f^* \le (1 - \mu/L)^k (f(\mathbf{x}_0) - f^*)$.

**Step 1 (strong convexity controls suboptimality by gradient norm).** Fix $\mathbf{x}$ and minimize
both sides of Definition 4 over $\mathbf{y}$. The right side is a quadratic in $\mathbf{y}$ minimized at
$\mathbf{y} = \mathbf{x} - \frac{1}{\mu}\nabla f(\mathbf{x})$, with minimum value
$f(\mathbf{x}) - \frac{1}{2\mu}\lVert \nabla f(\mathbf{x})\rVert^2$. The left side is minimized at
$f^*$. Hence

$$
f^* \ge f(\mathbf{x}) - \frac{1}{2\mu}\lVert \nabla f(\mathbf{x})\rVert^2 \quad \Longrightarrow \quad \lVert \nabla f(\mathbf{x})\rVert^2 \ge 2\mu\left(f(\mathbf{x}) - f^*\right)
$$

(This is exactly the PL inequality — strong convexity implies PL.)

**Step 2 (sufficient decrease).** From Proof 2, with $\alpha = 1/L$:

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \frac{1}{2L}\lVert \nabla f(\mathbf{x}_k)\rVert^2
$$

**Step 3 (combine).** Insert the bound from Step 1 into Step 2:

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \frac{\mu}{L}\left(f(\mathbf{x}_k) - f^*\right)
$$

**Step 4.** Subtract $f^*$ from both sides to obtain the one-step contraction of the suboptimality gap
$\delta_k = f(\mathbf{x}_k) - f^*$:

$$
\delta_{k+1} \le \left(1 - \frac{\mu}{L}\right)\delta_k
$$

**Step 5.** Iterate $k$ times:

$$
\boxed{f(\mathbf{x}_k) - f^* \le \left(1 - \frac{\mu}{L}\right)^k \left(f(\mathbf{x}_0) - f^*\right) = \left(1 - \frac{1}{\kappa}\right)^k \left(f(\mathbf{x}_0) - f^*\right)}
$$

Since $(1 - 1/\kappa)^k \le e^{-k/\kappa}$, accuracy $\epsilon$ requires $k \ge \kappa \log(\delta_0/\epsilon)$
iterations: the cost of gradient descent is *linear in the condition number*.

### Proof 5: The PL Inequality Implies Linear Convergence

**Claim.** If $f$ is $L$-smooth, bounded below with minimum $f^*$, and satisfies the PL inequality
$\frac{1}{2}\lVert \nabla f(\mathbf{x})\rVert^2 \ge \mu(f(\mathbf{x}) - f^*)$, then gradient descent
with $\alpha = 1/L$ converges linearly — no convexity required.

**Step 1.** Sufficient decrease (Proof 2, Step 2) holds for any $L$-smooth function:

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \frac{1}{2L}\lVert \nabla f(\mathbf{x}_k)\rVert^2
$$

**Step 2.** The PL inequality lower-bounds the payment: $\frac{1}{2}\lVert \nabla f(\mathbf{x}_k)\rVert^2 \ge \mu\left(f(\mathbf{x}_k) - f^*\right)$, so

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \frac{\mu}{L}\left(f(\mathbf{x}_k) - f^*\right)
$$

**Step 3.** Subtract $f^*$ and iterate, exactly as in Proof 4:

$$
\boxed{f(\mathbf{x}_k) - f^* \le \left(1 - \frac{\mu}{L}\right)^k\left(f(\mathbf{x}_0) - f^*\right) \quad \text{under PL alone}}
$$

**Why this matters.** The proof consumed only (a) the descent lemma and (b) the PL inequality. Convexity
appears nowhere. Modern analyses show that over-parameterized neural networks near initialization, matrix
factorization objectives, and consistent over-determined linear systems satisfy PL locally or globally —
explaining fast training on non-convex landscapes.

### Proof 6: Exact Quadratic Analysis — Optimal Step $2/(L+\mu)$ and Rate $\frac{\kappa-1}{\kappa+1}$

**Setting.** $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^T A\mathbf{x} - \mathbf{b}^T\mathbf{x}$ with
symmetric $A$, eigenvalues $\mu = \lambda_1 \le \dots \le \lambda_n = L$, $\mu \gt 0$; minimizer
$\mathbf{x}^* = A^{-1}\mathbf{b}$, gradient $\nabla f(\mathbf{x}) = A\mathbf{x} - \mathbf{b} = A(\mathbf{x} - \mathbf{x}^*)$.

**Step 1 (error dynamics are linear).** Let $\mathbf{e}_k = \mathbf{x}_k - \mathbf{x}^*$. Then

$$
\mathbf{e}_{k+1} = \mathbf{x}_k - \alpha A(\mathbf{x}_k - \mathbf{x}^*) - \mathbf{x}^* = (I - \alpha A)\,\mathbf{e}_k
$$

**Step 2 (diagonalize).** Write $A = Q\Lambda Q^T$ with orthonormal eigenvectors. In eigen-coordinates
$\tilde{\mathbf{e}} = Q^T\mathbf{e}$, each component evolves independently:
$\tilde{e}_{k+1,i} = (1 - \alpha\lambda_i)\,\tilde{e}_{k,i}$. The worst-case contraction factor is the
spectral radius

$$
\rho(\alpha) = \max_i \lvert 1 - \alpha\lambda_i\rvert = \max\left(\lvert 1 - \alpha\mu\rvert,\; \lvert 1 - \alpha L\rvert\right)
$$

since the maximum of a convex function of $\lambda$ over $[\mu, L]$ is attained at an endpoint.
Convergence for every $\mathbf{x}_0$ requires $\rho(\alpha) \lt 1$, i.e. $0 \lt \alpha \lt 2/L$.

**Step 3 (optimize the step).** $\lvert 1 - \alpha\mu\rvert$ increases in effect as $\alpha$ shrinks
while $\lvert 1 - \alpha L\rvert$ grows as $\alpha$ grows; the max is minimized where the two branches
are equal and of opposite sign:

$$
1 - \alpha\mu = \alpha L - 1 \quad \Longrightarrow \quad \alpha^* = \frac{2}{L + \mu}
$$

**Step 4 (optimal rate).** Substitute $\alpha^*$:

$$
\rho(\alpha^*) = 1 - \frac{2\mu}{L+\mu} = \frac{L - \mu}{L + \mu} = \frac{\kappa - 1}{\kappa + 1}
$$

**Step 5 (interpretation).** Orthogonality of $Q$ preserves norms, so

$$
\boxed{\lVert \mathbf{x}_{k+1} - \mathbf{x}^*\rVert \le \frac{\kappa - 1}{\kappa + 1}\,\lVert \mathbf{x}_k - \mathbf{x}^*\rVert \quad \text{with } \alpha^* = \frac{2}{L+\mu}}
$$

For $\kappa = 100$ the factor is $99/101 \approx 0.98$: about $230$ iterations per digit of accuracy.
The two extreme eigen-directions are simultaneously binding — the algebraic signature of zig-zagging:
the error component along $\lambda = L$ flips sign every step (oscillation across the valley) while the
component along $\lambda = \mu$ shrinks slowly (the crawl along the valley floor).

## 4. Computational & Algorithmic Insights

### 4.1 Choosing the Step Size in Practice

The theory prescribes $\alpha = 1/L$, but $L$ is rarely known. Practical strategies:

- **Backtracking (Armijo)**: start large, halve until sufficient decrease holds; adapts automatically to
  local curvature (full treatment in Topic 04).
- **Divergence threshold**: on a quadratic, any fixed $\alpha \ge 2/L$ diverges along the stiffest
  eigenvector since $\lvert 1 - \alpha L\rvert \ge 1$. In deep learning this appears as the loss
  exploding when the learning rate crosses roughly twice the inverse sharpness — the *edge of stability*.
- **Estimating $L$ cheaply**: for least squares, $L = \lambda_{\max}(X^T X)$ can be obtained by a few
  power-method iterations at $O(\text{nnz}(X))$ cost each; a Lipschitz estimate also follows from
  $\lVert \nabla f(\mathbf{x}) - \nabla f(\mathbf{y})\rVert / \lVert \mathbf{x} - \mathbf{y}\rVert$ samples.

Cost accounting: one gradient descent iteration is one gradient evaluation plus $O(n)$ vector work — no
linear algebra, no memory beyond a few vectors. This is why first-order methods own the large-scale regime.

### 4.2 Diagnosing Conditioning from Trajectories

The eigen-decomposition in Proof 6 turns convergence curves into spectral diagnostics:

- A **straight line on a semilog plot** of $f(\mathbf{x}_k) - f^*$ indicates linear convergence; its
  slope estimates $\log(1 - 1/\kappa)$, hence $\kappa$.
- An **initial fast drop followed by a long plateau** means the components along large eigenvalues
  died quickly and the small-$\lambda$ components dominate — a signature of large $\kappa$.
- **Oscillating iterates** (sign-alternating coordinates) reveal $\alpha \gt 1/L$ behavior along stiff
  directions: reduce the step or precondition.

**Preconditioning** replaces the update by $\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha P^{-1}\nabla f(\mathbf{x}_k)$
with $P \approx \nabla^2 f$: the effective condition number becomes that of $P^{-1/2} (\nabla^2 f) P^{-1/2}$.
Feature standardization, batch normalization, and adaptive per-coordinate step sizes (Adam's
$\sqrt{\hat{v}_k}$ denominator) are all industrial preconditioners in this precise sense.

### 4.3 Implementing Momentum Correctly

Two equivalent parameterizations of heavy-ball momentum appear in libraries:

$$
\mathbf{v}_{k+1} = \beta\mathbf{v}_k + \nabla f(\mathbf{x}_k), \qquad \mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\mathbf{v}_{k+1}
$$

versus the two-point form $\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\nabla f(\mathbf{x}_k) + \beta(\mathbf{x}_k - \mathbf{x}_{k-1})$.
They coincide, but note the **effective step size**: with velocity accumulation, the steady-state
response to a constant gradient is $\alpha/(1-\beta)$ times the raw step. Raising $\beta$ from $0.9$ to
$0.99$ multiplies the effective step by $10$ — the classic source of momentum-tuning explosions.

Practical rules distilled from the quadratic analysis:

- Tuned heavy-ball and Nesterov need $\beta \approx \left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2$; the popular default $\beta = 0.9$ corresponds to $\kappa \approx 400$.
- Nesterov's look-ahead gradient adds a damping term that suppresses the non-monotone spikes heavy-ball
  exhibits; on non-quadratic convex problems only Nesterov retains the $O(1/k^2)$ guarantee.
- As a discretized ODE, momentum is stable for $\alpha L \lt 2(1+\beta)$ — a *larger* stability region
  than plain gradient descent, another reason it tolerates aggressive steps.

## 5. Real-World Physics & AI/ML Applications

### 5.1 Machine Learning: Least Squares, Logistic Regression, Deep Nets

- **Linear regression.** For $f(\mathbf{w}) = \frac{1}{2}\lVert X\mathbf{w} - \mathbf{y}\rVert^2$ the
  Hessian is $X^T X$: gradient descent converges linearly with $\kappa = \lambda_{\max}(X^T X)/\lambda_{\min}(X^T X)$.
  Correlated features inflate $\kappa$ directly — the optimization-theoretic reason for feature
  standardization and whitening.
- **Logistic regression.** The negative log-likelihood is $L$-smooth with
  $L = \frac{1}{4}\lambda_{\max}(X^T X)$ (the sigmoid derivative never exceeds $1/4$) and convex, so
  Theorem 3 applies; with $\ell_2$ regularization $\frac{\lambda}{2}\lVert \mathbf{w}\rVert^2$ it becomes
  $\lambda$-strongly convex and Theorem 4 gives a linear rate.
- **Deep learning.** Losses are non-convex, so Theorem 2 ($O(1/\sqrt{k})$ to stationarity) is the honest
  guarantee; PL-type conditions explain why heavily over-parameterized networks nevertheless train at
  linear rates. Learning-rate schedules mirror the theory: warmup keeps $\alpha \lt 2/L_{\text{local}}$
  while curvature is large, decay refines the endgame.

### 5.2 Physics: Dissipative Dynamics and the Momentum Oscillator

Gradient flow $\dot{\mathbf{x}} = -\nabla f(\mathbf{x})$ is **overdamped dynamics**: the motion of a
particle in potential $f$ with friction so strong that inertia is negligible (Aristotelian mechanics).
Energy dissipates monotonically, which is why gradient descent is the numerical workhorse for:

- **Energy minimization** in molecular mechanics and elasticity — relaxing atomic positions or mesh
  nodes to equilibrium is literally gradient descent on a potential energy surface.
- **Heavy-ball dynamics.** Polyak's method discretizes $m\ddot{\mathbf{x}} + c\dot{\mathbf{x}} = -\nabla f(\mathbf{x})$,
  a damped oscillator. On a quadratic mode with curvature $\lambda$, the scalar dynamics are
  $m\ddot{e} + c\dot{e} + \lambda e = 0$: underdamped modes ring (oscillating loss curves), overdamped
  modes crawl, and **critical damping** — friction matched to curvature — is exactly the optimal-momentum
  tuning $\beta = \left((\sqrt{\kappa}-1)/(\sqrt{\kappa}+1)\right)^2$.
- **Nesterov's ODE.** In the continuum limit Nesterov's method solves
  $\ddot{x} + \frac{3}{t}\dot{x} + \nabla f(x) = 0$ (Su-Boyd-Candès): a vanishing-friction oscillator
  whose $3/t$ damping is precisely strong enough to secure the $O(1/t^2)$ energy decay — acceleration is
  a statement about carefully scheduled friction.

### 5.3 Case Study: The Rosenbrock Valley

The Rosenbrock function $f(x, y) = (1-x)^2 + 100(y - x^2)^2$ is the canonical stress test for this
module's theory. At the minimizer $(1,1)$ the Hessian is

$$
\nabla^2 f(1,1) = \begin{bmatrix} 802 & -400 \\ -400 & 200 \end{bmatrix}
$$

with eigenvalues $\lambda \approx 1001.6$ and $\lambda \approx 0.4$, so $\kappa \approx 2500$: the
optimal-step contraction $\frac{\kappa-1}{\kappa+1} \approx 0.9992$ predicts thousands of iterations per
digit. Gradient descent first plunges into the parabolic valley $y \approx x^2$ (killing the stiff
mode), then inches along its curved floor — exactly the fast-drop-then-plateau signature of Section 4.2.
Momentum cuts the iteration count roughly by $\sqrt{\kappa} \approx 50$, and the curvature-aware methods
of Topic 04 (Newton, BFGS) dispatch the valley in a handful of steps. The companion notebook
[computation.ipynb](../computation.ipynb) runs this experiment numerically.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Location |
|---|---|---|
| Descent methods, condition number analysis | Boyd & Vandenberghe, *Convex Optimization* | Chapter 9.2-9.3 |
| Steepest descent on quadratics, rate $\frac{\kappa-1}{\kappa+1}$ | Nocedal & Wright, *Numerical Optimization* (2nd ed.) | Chapter 3.3 |
| Gradient method convergence, step-size rules | Bertsekas, *Nonlinear Programming* (3rd ed.) | Chapter 1.2-1.3 |
| $L$-smoothness, $O(1/k)$ and linear rates, lower bounds, acceleration | Nesterov, *Lectures on Convex Optimization* (2nd ed.) | Chapter 2.1-2.2 |
| Heavy-ball method and its quadratic analysis; PL inequality | Polyak, *Introduction to Optimization* | Chapter 3 |
| Dimension-free proofs (template for Proofs 2-4) | Bubeck, *Convex Optimization: Algorithms and Complexity* | Chapter 3 |
| PL implies linear convergence (Proof 5) | Karimi, Nutini & Schmidt, ECML-PKDD 2016 | Theorem 1 |
| Nesterov ODE, momentum as damped oscillator | Su, Boyd & Candès, JMLR 17 (2016); Goh, *Distill* (2017) | Full papers |

**Suggested reading order.** Start with Boyd & Vandenberghe Chapter 9 for geometry and examples; then
Nesterov Chapter 2 for the clean smooth/strongly-convex rate theory reproduced here; Nocedal & Wright
Chapter 3.3 for the exact quadratic analysis; finally Polyak Chapter 3 and the Distill article for
momentum. The exercises notebook in this folder applies every theorem above to concrete spectra,
learning-rate thresholds, and physical analogies.